# TI-City 16b — Phase 0 evaluation & baseline anchor (batch R125–R128)

Evaluates the completed calibration batch (R125 = `s0`, R126 = `s_random`, R127 = `s_edge`, R128 = `s_roadbound`, seeds 11/22/33) against the observed 2000–2025 new-urban mask, then anchors `s0` to the baselines.

**Protocol (agreed):**
- All loading and metrics via the validation module (`ti_city_10_calibration_fuzzy_fom_validation_module.py`, commit 192f1b0 or later): `load_ascii_files` with `seeds_list` returns `{scenario: {seed: {tick: array}}}`, and `evaluate_scenario_fom` evaluates it natively with its seed branch (rows carry a `Seed` column).
- Ensemble **means** on BOTH sides (model and baselines) per scenario before `normalize_against_baselines` (it takes the max over all rows of a baseline; unaggregated per-seed rows would normalise against the strongest single seed instead of the ensemble mean).
- `per_tick='final'`, `variant='hits_only'`, `cell_km=0.150`, `start_year=2000`, tolerances (0, 1, 2, 3, 5).

# Imports

In [1]:
import sys
from pathlib import Path

import pandas as pd
import geopandas as gpd

import os
from pathlib import Path
#import fnmatch
#import shutil

import numpy as np
import random 
import matplotlib.pyplot as plt
#import matplotlib.colors as mcolors
#import matplotlib.patches as mpatches
import natsort

#from shapely.geometry.polygon import Polygon
#from shapely.geometry import MultiLineString, LineString
#import json

import rasterio
from rasterio.mask import mask
from rasterio.plot import show, show_hist, plotting_extent
from rasterio.warp import reproject, calculate_default_transform, Resampling
from rasterio.features import dataset_features
import rasterstats
#from libpysal.weights import lat2W, Rook
#from esda.moran import Moran

from ti_city_00_helper_functions import *
from ti_city_10_calibration_fuzzy_fom_validation_module import (
    restrict_mask,                        # drop initial-urban / invalid cells
    observed_new_from_rasters,            # obs new-urban from urban_late & urban_early arrays
    load_ascii_files,                     # run-folder ASCII loader: {scenario: {seed: {tick: array}}}
    evaluate_scenario_fom,                # FoM table incl. the native seed branch
    normalize_against_baselines,          # skill vs baselines (anchoring)
    null_fom_baseline,                   # chance floor for sanity checks
)

Cannot find header.dxf (GDAL_DATA is not defined)


# init

In [2]:
case_city = 'JAK' #MUM MAN JAK
drop_path = r'C:\\Sciebo\\05_GIS\\' # adjust this to match the path on your computer, e.g. 'D:\\Dropbox\\x\\PostDoc\\GIS' or 'C:\\Users\\x\\Dropbox\\x\\PostDoc\\GIS'
external_data_path = Path(r'C:\\Sciebo\\00_data') # adjust this to match the path on your computer, e.g. 'D:\\Dropbox\\x\\PostDoc\\00_data' or 'C:\\Users\\x\\Dropbox\\x\\PostDoc\\00_data'
## ascii150_arcgis_path = Path(r"D:\\Dropbox\\x\\PostDoc\ASCII_150m") deprecated, ignore

#input a vector and a raster file for each city
AOI_path = Path(f'{drop_path}\\{case_city}\\processed\\')
AOI_file = f'{case_city}_LIM_AOI_reference_150m_A.shp' # APS: updated AOI 01.08.2024

AOI_gdf = gpd.read_file(AOI_path / AOI_file) #.to_crs(epsg=4326)
AOI_gdf_4326 = AOI_gdf.to_crs(epsg=4326) 

ref_raster_path = Path(f'../data/processed/{case_city}_LIM_reference_AOI_150m.tif')
ref_raster_150_path = Path(f'../data/processed/{case_city}_LIM_reference_AOI_150m.tif')

raw_path = Path(f'\\{case_city}\\raw\\')
interim_path = Path(drop_path + f'{case_city}\\interim\\')
processed_path = Path(drop_path + f'{case_city}\\processed\\')
external_path = Path(drop_path + f'{case_city}\\external\\')
model_inputs_30m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_30m\\')
model_inputs_150m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_150m\\')
model_inputs_SLEUTH_150m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_SLEUTH_150m')
model_outputs_path = Path(drop_path + f'{case_city}\\model_outputs\\') 
ascii150_path = Path(drop_path + f'\\{case_city}\\model_inputs\\ASCII_150m')
ascii150_SLEUTH_path = Path(drop_path + f'\\{case_city}\\model_inputs\\ASCII_SLEUTH_150m')
ti_city_model_path = Path(f'../model/{case_city}/')
ti_city_in_path = Path(f'../model/{case_city}/data/in/')
ti_city_out_path = Path(f'../model/{case_city}/data/out/')
ref_raster_sleuth_path = model_inputs_SLEUTH_150m_path / (case_city + '_URB_SLEUTH_input_2022.tif' )

if case_city == 'MUM':
    gadm_var = 'NAME_3'
    calibration_years = [2015,2025]
    simulation_years = [2025, 2050]

if case_city == 'MAN':    
    gadm_var = 'NAME_2'
    calibration_years = [2015,2025]
    simulation_years = [2025, 2050]

if case_city == 'JAK':
    gadm_var = 'NAME_3'
    calibration_years = [2000,2025]
    simulation_years = [2025, 2050]

# read the reference raster
with rasterio.open(ref_raster_path,'r') as src: # APS 20.08.2025 using the 150 m raster
    ref_raster = src
    ref_meta = src.meta
    ref_height, ref_width, ref_area = get_transform(ref_raster)

# suppress deprecation warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

# first, let's inform the scenario folders
# APS: 15.05.2026 I updated the folder and file naming convention to allow for individual results per tick
# example: "D:\GIT\ti-city-model\model\JAK\data\out\R037\expansion_income_t0.asc"


# let's build dictionaries to support the loading functions
agent_class_dict = {'na':0,'low': 1, 'mid': 2, 'high': 3, 'deprived': 4, 'urban_2022': 5}
#APS 11.08.2026 previously: ['std', 'ssp1', 'ssp2', 'ssp3']
#APS 22.10.2026 previously: ['S1', 'S2', 'S3', 'S4'] 
ascii_files = [] # initialize the list of ASCII files

c:\Users\santos\.conda\envs\x\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files.  Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


## 1. Batch configuration

Run map for batch R125–R128 (matches `docs/JAK_TA-model_manifest_R125_R128.csv`).

In [3]:
run_map = {
    "R125": {"scenario": "s0",          "role": "model"},
    "R126": {"scenario": "s_random",    "role": "baseline"},
    "R127": {"scenario": "s_edge",      "role": "baseline"},
    "R128": {"scenario": "s_roadbound", "role": "baseline"},
}

#model_scenarios    = [r["scenario"] for r in run_map.values() if r["role"] == "model"]
#baseline_scenarios = [r["scenario"] for r in run_map.values() if r["role"] == "baseline"]
scenario_path = {r["scenario"]: f"{run_id}/" for run_id, r in run_map.items()}

seeds_list    = ["seed11", "seed22", "seed33"]   # folder names under data/out/R<run>/
outputs_dir   = ti_city_out_path
reference_dir = ti_city_in_path   # urban_early / urban_late
docs_dir      = ti_city_model_path / "../../docs"

# FoM settings (agreed protocol)
fom_start_year = calibration_years[0]
fom_end_year   = calibration_years[1]
fom_cell_km    = 0.150
fom_variant    = "hits_only"

#print("model:", model_scenarios, "| baselines:", baseline_scenarios)
print("scenario_path:", scenario_path)

scenario_path: {'s0': 'R125/', 's_random': 'R126/', 's_edge': 'R127/', 's_roadbound': 'R128/'}


## 2. Load the run rasters

`load_ascii_files` (module loader) with `seeds_list` produces the seed-nested dict:

- `results_dict`  = `{s0: {seed11/22/33: {tick: array}}}`
- `baseline_dict` = `{s_random, s_edge, s_roadbound: {seed: {tick: array}}}`

`load_mode='final'` reads only the last tick (`expansion_income_t025.asc`), which is all `per_tick='final'` evaluation needs.

In [4]:
scenario_path_dict = {'s0': 'R125/', 's_random': 'R126/', 's_edge': 'R127/', 's_roadbound': 'R128/'} # consider the output_folder
results_dict = load_ascii_files(
    scenarios = ['s0'],
    scenario_path = scenario_path_dict,
    seeds_list = seeds_list,
    load_mode = "final",
    results_folder = outputs_dir,
    file_name_template = "expansion_income_t",
    export_tiff = False,
    ref_raster_path = reference_dir / "urban_early.asc",
    case_city = "JAK",
    silent = False,
)

baseline_dict = load_ascii_files(
    scenarios = ['s_random', 's_edge', 's_roadbound'],
    scenario_path = scenario_path_dict,
    seeds_list = seeds_list,
    load_mode = "final",
    results_folder = outputs_dir,
    file_name_template = "expansion_income_t",
    export_tiff = False,
    ref_raster_path = reference_dir / "urban_early.asc",
    case_city = "JAK",
    silent = False,
)

for sc, seed_dict in {**results_dict, **baseline_dict}.items():
    for seed, series in seed_dict.items():
        assert series, f"no ticks loaded for {sc} {seed}"
        print(sc, seed, "ticks:", sorted(series))

Processing content from: s0 and seed: seed11 at ..\model\JAK\data\out\R125\seed11
Loading only the final tick for scenario: s0 and seed: seed11
Extracted key: 025 from file: expansion_income_t025.asc
Processing content from: s0 and seed: seed22 at ..\model\JAK\data\out\R125\seed22
Loading only the final tick for scenario: s0 and seed: seed22
Extracted key: 025 from file: expansion_income_t025.asc
Processing content from: s0 and seed: seed33 at ..\model\JAK\data\out\R125\seed33
Loading only the final tick for scenario: s0 and seed: seed33
Extracted key: 025 from file: expansion_income_t025.asc
Processing content from: s_random and seed: seed11 at ..\model\JAK\data\out\R126\seed11
Loading only the final tick for scenario: s_random and seed: seed11
Extracted key: 025 from file: expansion_income_t025.asc
Processing content from: s_random and seed: seed22 at ..\model\JAK\data\out\R126\seed22
Loading only the final tick for scenario: s_random and seed: seed22
Extracted key: 025 from file: ex

## 3. Observed new-urban mask (2000–2025)

Built from the project's reference rasters with the module function `observed_new_from_rasters`:

- `obs_new` = urban 2025 AND NOT urban 2000
- `initial_urban` = urban 2000

Both inputs must be on the 150 m AOI reference grid (680 x 772 for JAK).

In [5]:
urban_early_path = reference_dir / "urban_early.asc"
urban_late_path  = reference_dir / "urban_late.asc"

with rasterio.open(urban_early_path, "r") as src:
    urban_early = src.read(1)
    urban_early_nodata = src.nodata
with rasterio.open(urban_late_path, "r") as src:
    urban_late = src.read(1)
    urban_late_nodata = src.nodata

print("urban_early:", urban_early.shape, "nodata:", urban_early_nodata)
print("urban_late: ", urban_late.shape, "nodata:", urban_late_nodata)

obs_new = observed_new_from_rasters(
    urban_late = urban_late,
    urban_early = urban_early,
    valid_mask = None,      # pass an AOI/nodata mask if the reference rasters carry positive nodata
)
initial_urban = urban_early > 0

print(f"observed new-urban cells: {int(obs_new.sum()):,}")
print(f"initial urban cells:       {int(initial_urban.sum()):,}")

urban_early: (680, 772) nodata: -9999.0
urban_late:  (680, 772) nodata: -9999.0
observed new-urban cells: 29,656
initial urban cells:       126,651


## 4. Per-seed FoM evaluation

One `evaluate_scenario_fom` call per side, passing `results_dict` exactly as `load_ascii_files` returned it plus the matching `seeds_list`. With a non-None `seeds_list` the function uses its seed branch (`{scenario: {seed: {tick: array}}}`) and emits one row per scenario x seed x tick with a `Seed` column.

In [6]:
fom_per_seed = evaluate_scenario_fom(
    results_dict = results_dict,
    obs_new = obs_new,
    seeds_list = seeds_list,
    initial_urban = initial_urban,
    valid_mask = None,
    tolerances = (0, 1, 2, 3, 5),
    variant = fom_variant,
    per_tick = "final",
    start_year = fom_start_year,
    case_city = "JAK",
    cell_km = fom_cell_km,
    silent = True,
)

baseline_fom_per_seed = evaluate_scenario_fom(
    results_dict = baseline_dict,
    obs_new = obs_new,
    seeds_list = seeds_list,
    initial_urban = initial_urban,
    valid_mask = None,
    tolerances = (0, 1, 2, 3, 5),
    variant = fom_variant,
    per_tick = "final",
    start_year = fom_start_year,
    case_city = "JAK",
    cell_km = fom_cell_km,
    silent = True,
)

display(fom_per_seed[["Scenario", "Seed", "Year", "N_sim", "FoM",
                     "FuzzyFoM_t1", "FuzzyFoM_t2", "FuzzyFoM_t3", "FuzzyFoM_t5"]])

,Scenario,Seed,Year,N_sim,FoM,FuzzyFoM_t1,FuzzyFoM_t2,FuzzyFoM_t3,FuzzyFoM_t5
0,s0,seed11,2025,29647,0.307818,0.419054,0.468762,0.497916,0.532848
1,s0,seed22,2025,29647,0.307328,0.418081,0.468983,0.498898,0.533045
2,s0,seed33,2025,29647,0.306809,0.418643,0.469833,0.498832,0.532107


## 5. Ensemble means (model AND baselines)

Average the per-seed rows per scenario. This is the critical step before normalisation: `normalize_against_baselines` takes the max over all rows of a baseline, so unaggregated per-seed rows would normalise against the strongest single seed instead of the ensemble mean.

In [7]:
def _ensemble_means(fom_table=None, label=""):
    df = fom_table.copy()
    group_keys = ["Scenario", "Tick", "Year"]
    value_cols = [c for c in df.columns if c not in group_keys + ["Seed"]]
    df[value_cols] = df[value_cols].apply(pd.to_numeric, errors="coerce")
    means = df.groupby(group_keys, as_index=False)[value_cols].mean()
    print(f"{label}: {len(fom_table)} per-seed rows -> {len(means)} scenario rows")
    return means

fom_means      = _ensemble_means(fom_table=fom_per_seed,          label="model")
baseline_means = _ensemble_means(fom_table=baseline_fom_per_seed, label="baselines")

display(baseline_means[["Scenario", "Year", "N_sim", "FoM",
                        "FuzzyFoM_t1", "FuzzyFoM_t2", "FuzzyFoM_t3", "FuzzyFoM_t5"]])

model: 3 per-seed rows -> 1 scenario rows
baselines: 9 per-seed rows -> 3 scenario rows


,Scenario,Year,N_sim,FoM,FuzzyFoM_t1,FuzzyFoM_t2,FuzzyFoM_t3,FuzzyFoM_t5
0,s_edge,2025,29653.0,0.334617,0.526108,0.590067,0.619479,0.645942
1,s_random,2025,29653.0,0.061010,0.292864,0.438621,0.488760,0.514145
2,s_roadbound,2025,29656.0,0.132262,0.405820,0.496917,0.529427,0.550394


## 6. Anchoring: normalise against the baselines

`normalize_against_baselines` adds `*_skill` columns (vs the strongest baseline per metric) and one `*_vs_<baseline>` column per baseline, so we can read WHICH mechanism binds.

In [8]:
fom_anchor = normalize_against_baselines(
    fom_table = fom_means,
    baseline_fom_table = baseline_means,
    perfect = 1.0,
    per_baseline = True,
    group_col = "Tick",
)

anchor_cols = ["Scenario", "Year", "N_sim", "QuantityRatio", "FoM",
               "FuzzyFoM_t1", "FuzzyFoM_t2", "FuzzyFoM_t3", "FuzzyFoM_t5"]
display(fom_anchor[[c for c in anchor_cols if c in fom_anchor.columns]
                   + [c for c in fom_anchor.columns if c.endswith("_skill")]])

,Scenario,Year,N_sim,QuantityRatio,FoM,FuzzyFoM_t1,FuzzyFoM_t2,FuzzyFoM_t3,FuzzyFoM_t5,FuzzyFoM_t1_skill,FuzzyFoM_t2_skill,FuzzyFoM_t3_skill,FuzzyFoM_t5_skill,FoM_skill
0,s0,2025,29647.0,0.999697,0.307318,0.418593,0.469192,0.498549,0.532667,-0.226876,-0.294866,-0.317801,-0.319934,-0.041027


## 7. Anchor diagnosis: which baseline binds per metric

The binding baseline for a metric is the one whose FoM the model must beat first (its `<metric>_vs_<baseline>` skill is 0).

In [9]:
metric_cols = ["FoM", "FuzzyFoM_t1", "FuzzyFoM_t2", "FuzzyFoM_t3", "FuzzyFoM_t5"]
baseline_names = sorted(baseline_means["Scenario"].unique())

rows = []
for _, r in fom_anchor.iterrows():
    for metric in metric_cols:
        vs = {b: r.get(f"{metric}_vs_{b}", np.nan) for b in baseline_names}
        binding = min(vs, key=vs.get)
        rows.append({
            "Scenario": r["Scenario"], "Year": r["Year"], "Metric": metric,
            "Model_mean": r[metric],
            "Binding_baseline": binding,
            "Best_baseline_value": baseline_means.loc[baseline_means["Scenario"] == binding, metric].max(),
            "Skill": r[f"{metric}_skill"],
        })
anchor_diag = pd.DataFrame(rows)
display(anchor_diag)

for metric in metric_cols:
    sub = anchor_diag[anchor_diag["Metric"] == metric]
    print(f"{metric}: binding baseline = {sub['Binding_baseline'].iloc[0]} "
          f"(model {sub['Model_mean'].iloc[0]:.3f} vs baseline {sub['Best_baseline_value'].iloc[0]:.3f}, "
          f"skill {sub['Skill'].iloc[0]:+.3f})")

,Scenario,Year,Metric,Model_mean,Binding_baseline,Best_baseline_value,Skill
0,s0,2025,FoM,0.307318,s_edge,0.334617,-0.041027
1,s0,2025,FuzzyFoM_t1,0.418593,s_edge,0.526108,-0.226876
2,s0,2025,FuzzyFoM_t2,0.469192,s_edge,0.590067,-0.294866
3,s0,2025,FuzzyFoM_t3,0.498549,s_edge,0.619479,-0.317801
4,s0,2025,FuzzyFoM_t5,0.532667,s_edge,0.645942,-0.319934


FoM: binding baseline = s_edge (model 0.307 vs baseline 0.335, skill -0.041)
FuzzyFoM_t1: binding baseline = s_edge (model 0.419 vs baseline 0.526, skill -0.227)
FuzzyFoM_t2: binding baseline = s_edge (model 0.469 vs baseline 0.590, skill -0.295)
FuzzyFoM_t3: binding baseline = s_edge (model 0.499 vs baseline 0.619, skill -0.318)
FuzzyFoM_t5: binding baseline = s_edge (model 0.533 vs baseline 0.646, skill -0.320)


## 8. Sanity check: random-allocation chance floor

`null_fom_baseline` scatters `n_obs` cells over the eligible (non-initial-urban) cells. The ABM baselines should sit above this floor; the model should sit above the baselines.

In [10]:
eligible_mask = restrict_mask(
    mask = np.ones_like(initial_urban, dtype=bool),
    initial_urban = initial_urban,
    valid_mask = None,
)

null_table = null_fom_baseline(
    obs_new = np.asarray(obs_new, dtype=bool),
    eligible_mask = eligible_mask,
    n_seeds = 5,
    tolerances = (0, 1, 2, 3, 5),
    variant = fom_variant,
    seed = 42,
)
display(null_table)

,Scenario,N_sim,N_obs,FoM_t0,FoM_t1,FoM_t2,FoM_t3,FoM_t5
0,NULL_random,29656,29656,0.037798,0.213044,0.376237,0.459223,0.504593


## 9. Export the anchor tables

Writes the Phase 0 anchor result next to the batch manifest in `docs/` so it can be appended to `TI_model_runs.xlsx` / the cumulative results table.

In [11]:
export_path = docs_dir / "JAK_TA-model_BS_phase0_anchor_R125-128.csv"
fom_anchor.to_csv(export_path, index=False)
anchor_diag.to_csv(docs_dir / "JAK_TA-model_BS_phase0_anchor_diag_R125-128.csv", index=False)
print("exported:", export_path)

exported: ..\model\JAK\..\..\docs\JAK_TA-model_BS_phase0_anchor_R125-128.csv
